##  a Python A2A server acts as a central hub for AI agents, enabling them to communicate and collaborate using the A2A protocol and potentially MCP, facilitating the development of multi-agent systems. 

In [17]:
!pip install -r requirements.txt

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.10; 0.2.0 Requires-Python >=3.10; 0.3.0 Requires-Python >=3.10; 0.3.1 Requires-Python >=3.10; 0.3.2 Requires-Python >=3.10; 0.3.3 Requires-Python >=3.10; 0.3.4 Requires-Python >=3.10; 0.3.5 Requires-Python >=3.10; 0.4.0 Requires-Python >=3.10; 0.4.1 Requires-Python >=3.10; 1.0 Requires-Python >=3.10; 2.0.0 Requires-Python >=3.10; 2.1.0 Requires-Python >=3.10; 2.1.1 Requires-Python >=3.10; 2.1.2 Requires-Python >=3.10; 2.2.0 Requires-Python >=3.10; 2.2.1 Requires-Python >=3.10; 2.2.10 Requires-Python >=3.10; 2.2.2 Requires-Python >=3.10; 2.2.3 Requires-Python >=3.10; 2.2.4 Requires-Python >=3.10; 2.2.5 Requires-Python >=3.10; 2.2.6 Requires-Python >=3.10; 2.2.7 Requires-Python >=3.10; 2.2.8 Requires-Python >=3.10; 2.2.9 Requires-Python >=3.10; 2.3.0 Requires-Python >=3.10; 2.3.0rc1 Requires-Python >=3.10; 2.3.1 Requires-Python >=3.10; 2.3.2 Requires-Python >=3.10; 2.3.3 Requires-Pyth

In [19]:
!pip install yfinance

                                              0.0/117.9 kB ? eta -:--:--
     -------------------------------------- 117.9/117.9 kB 6.7 MB/s eta 0:00:00
  Using cached requests-2.32.3-py3-none-any.whl (64 kB)
                                              0.0/3.0 MB ? eta -:--:--
     ---------                                0.7/3.0 MB 14.6 MB/s eta 0:00:01
     ------------------                       1.4/3.0 MB 14.9 MB/s eta 0:00:01
     -------------------------------          2.4/3.0 MB 17.1 MB/s eta 0:00:01
     ---------------------------------------- 3.0/3.0 MB 16.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
                                              0.0/1.3 MB ? eta -:--:--
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pypark 0.0.10 requires requests~=2.25.0, but you have requests 2.32.3 which is incompatible.

[notice] A new release of pip is available: 23.1.2 -> 25.0.1
[notice] To update, run: c:\users\akarsh\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip


In [1]:
import os
import re
import yfinance
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from python_a2a import OpenAIA2AServer, run_server
from python_a2a import AgentCard, AgentSkill
from python_a2a.mcp import FastMCP
from python_a2a.langchain import to_langchain_agent, to_langchain_tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool, AgentType

# Set your OpenAI API key directly here
OPENAI_API_KEY = "Add Your OpenAI API Key"  # Replace with your actual key
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

ModuleNotFoundError: No module named 'yfinance'

# Agent-1 Financial Expert -(Stock Market Expert)

In [ ]:
# Define the financial expert agent's capabilities
agent_card = AgentCard(
    name="Stock Market Expert",
    description="Expert in market trends, fundamentals, and investment strategies.",
    url="http://localhost:5000",
    version="1.0.0",
    skills=[
        AgentSkill(
            name="Market Analysis",
            description="Analyze overall market sentiment and key indicators.",
            examples=["What's the current market sentiment?", "Impact of interest rates on tech stocks?"]
        ),
        AgentSkill(
            name="Investment Strategies",
            description="Describe risk management and portfolio diversification.",
            examples=["How to diversify my portfolio?", "Explanation of dollar-cost averaging."]
        ),
        AgentSkill(
            name="Company Analysis",
            description="Interpret financial ratios and company fundamentals.",
            examples=["How to read P/E ratios?", "Key metrics for evaluating growth stocks."]
        )
    ]
)

print("Agent profile created successfully!")
print(f"Agent Name: {agent_card.name}")
print(f"Skills: {[skill.name for skill in agent_card.skills]}")

# Creating A2A server

# Create the A2A server instance
a2a_server = OpenAIA2AServer(
    api_key=OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0.0,
    system_prompt="You are a stock market and financial analysis expert. Provide factual, concise insights based on available data and market knowledge."
)

print("A2A Server initialized successfully!")

# Run this server

In [ ]:
import threading
import time

def start_a2a_server():
    """Start A2A server in background thread"""
    try:
        run_server(a2a_server, host='0.0.0.0', port=5000)
    except Exception as e:
        print(f"A2A Server error: {e}")

# Start server in background thread
a2a_thread = threading.Thread(target=start_a2a_server, daemon=True)
a2a_thread.start()

# Give server time to start
time.sleep(3)
print("A2A Server started on http://localhost:5000")

# Create MCP server

In [ ]:
# Create MCP server for financial tools
mcp_server = FastMCP(
    name="FinanceTools",
    description="Tools for retrieving stock data and financial news."
)

print("MCP Server initialized!")

# Logic given as a function to to above ai agent 

# SubAgent-1 stock_data

In [ ]:
@mcp_server.tool(
    name="stock_data",
    description="Fetch current metrics and data for stocks by ticker symbols or company names."
)
def stock_data(input_str=None, **kwargs):
    """Fetch stock data using yfinance"""
    input_str = kwargs.get('input', input_str)
    if not input_str:
        return {"error": "No input provided."}
    
    # Extract ticker symbols
    tickers = []
    if ',' in input_str:
        tickers = [t.strip().upper() for t in input_str.split(',')]
    else:
        tickers = [w.upper() for w in re.findall(r"\b[A-Za-z]{1,5}\b", input_str)]
    
    # Handle common company names
    common_names = {
        'apple': 'AAPL', 
        'nvidia': 'NVDA', 
        'microsoft': 'MSFT',
        'google': 'GOOGL',
        'amazon': 'AMZN',
        'tesla': 'TSLA'
    }
    
    if not tickers:
        for name, ticker in common_names.items():
            if name in input_str.lower(): 
                tickers.append(ticker)
    
    results = {}
    for ticker in tickers:
        try:
            tk = yfinance.Ticker(ticker)
            hist = tk.history(period="1mo")
            
            if hist.empty:
                results[ticker] = {"error": "No data available."}
                continue
            
            first_day = hist.iloc[0]
            last_day = hist.iloc[-1]
            price_change = float(last_day['Close'] - first_day['Close'])
            pct_change = price_change / float(first_day['Close']) * 100
            
            info = tk.info
            
            summary = {
                "latest_price": float(last_day['Close']),
                "price_change": price_change,
                "percent_change": pct_change,
                "52_week_high": info.get('fiftyTwoWeekHigh'),
                "52_week_low": info.get('fiftyTwoWeekLow'),
                "market_cap": info.get('marketCap'),
                "pe_ratio": info.get('trailingPE'),
                "volume": int(last_day['Volume'])
            }
            results[ticker] = summary
            
        except Exception as e:
            results[ticker] = {"error": f"Failed to fetch data: {str(e)}"}
    
    return results

print("Stock data fetcher tool created!")

# # SubAgent-2 web_scraper

In [ ]:
@mcp_server.tool(
    name="web_scraper",
    description="Scrape latest financial headlines and company snapshot from Finviz."
)
def web_scraper(input_str=None, **kwargs):
    """Scrape financial news and data from Finviz"""
    ticker = (kwargs.get('input') or input_str or '').strip().upper()
    
    if not ticker:
        return {"error": "No ticker symbol provided."}
    
    url = f"https://finviz.com/quote.ashx?t={ticker.lower()}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract news items
        news_items = []
        news_table = soup.find('table', {'id': 'news-table'})
        if news_table:
            for row in news_table.find_all('tr')[:5]:  # Get top 5 news items
                cells = row.find_all('td')
                if len(cells) >= 2:
                    date_cell = cells[0]
                    title_cell = cells[1]
                    
                    link_tag = title_cell.find('a')
                    if link_tag:
                        title = link_tag.text.strip()
                        link = link_tag.get('href', '')
                        if link and not link.startswith('http'):
                            link = urljoin(url, link)
                        
                        news_items.append({
                            "date": date_cell.text.strip(),
                            "title": title,
                            "link": link
                        })
        
        # Extract company snapshot data
        snapshot_details = {}
        snapshot_table = soup.find('table', {'class': 'snapshot-table2'})
        if snapshot_table:
            for row in snapshot_table.find_all('tr'):
                cells = row.find_all('td')
                for i in range(0, len(cells), 2):
                    if i + 1 < len(cells):
                        key = cells[i].text.strip()
                        value = cells[i + 1].text.strip()
                        snapshot_details[key] = value
        
        return {
            "ticker": ticker,
            "news_items": news_items,
            "snapshot": snapshot_details
        }
        
    except Exception as e:
        return {"error": f"Failed to scrape data: {str(e)}"}

print("Financial news scraper tool created!")

# Run MCP server

In [ ]:
def start_mcp_server():
    """Start MCP server in background thread"""
    try:
        mcp_server.run(host='0.0.0.0', port=6000)
    except Exception as e:
        print(f"MCP Server error: {e}")

# Start MCP server in background thread
mcp_thread = threading.Thread(target=start_mcp_server, daemon=True)
mcp_thread.start()

# Give server time to start
time.sleep(3)
print("MCP Server started on http://localhost:6000")

# Convert both A2A agent & MCP tools to LangChain

In [ ]:
# Convert A2A agent to LangChain
try:
    a2a_agent = to_langchain_agent("http://localhost:5000")
    print("A2A agent converted to LangChain successfully!")
except Exception as e:
    print(f"Error converting A2A agent: {e}")

# Convert MCP tools to LangChain
try:
    stock_tool = to_langchain_tool("http://localhost:6000", "stock_data")
    news_tool = to_langchain_tool("http://localhost:6000", "web_scraper")
    print("MCP tools converted to LangChain successfully!")
except Exception as e:
    print(f"Error converting MCP tools: {e}")

In [ ]:
def ask_expert(query):
    """Ask the financial expert agent"""
    try:
        result = a2a_agent.invoke(query)
        return result.get('output', 'No response from expert')
    except Exception as e:
        return f"Error asking expert: {str(e)}"

def fetch_stock_data(query):
    """Fetch stock data"""
    try:
        return stock_tool.invoke(query)
    except Exception as e:
        return f"Error fetching stock data: {str(e)}"

def fetch_financial_news(query):
    """Fetch financial news"""
    try:
        return news_tool.invoke(query)
    except Exception as e:
        return f"Error fetching news: {str(e)}"

print("Wrapper functions defined!")

# Create function to call in order

In [ ]:
def ask_expert(query):
    """Ask the financial expert agent"""
    try:
        result = a2a_agent.invoke(query)
        return result.get('output', 'No response from expert')
    except Exception as e:
        return f"Error asking expert: {str(e)}"

def fetch_stock_data(query):
    """Fetch stock data"""
    try:
        return stock_tool.invoke(query)
    except Exception as e:
        return f"Error fetching stock data: {str(e)}"

def fetch_financial_news(query):
    """Fetch financial news"""
    try:
        return news_tool.invoke(query)
    except Exception as e:
        return f"Error fetching news: {str(e)}"

print("Wrapper functions defined!")

#### We create LangChain Tool objects because they are the fundamental mechanism through which a LangChain agent can interact with the outside world and perform specific actions beyond just generating text.

In [ ]:
# Create LangChain Tool objects
tools = [
    Tool(
        name="StockExpert",
        func=ask_expert,
        description="Ask financial questions to a stock market expert. Use for market analysis, investment strategies, and financial advice."
    ),
    Tool(
        name="StockData",
        func=fetch_stock_data,
        description="Retrieve current stock metrics and data. Input should be ticker symbols or company names."
    ),
    Tool(
        name="FinancialNews",
        func=fetch_financial_news,
        description="Get latest financial headlines and company snapshots. Input should be a ticker symbol."
    )
]

print(f"Created {len(tools)} LangChain tools!")

# initialise agent

In [ ]:
# Initialize the main LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    api_key=OPENAI_API_KEY
)

# Create the meta-agent
meta_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    max_iterations=5,
    early_stopping_method="generate"
)

print("Meta-agent initialized successfully!")

Test various types of queries

In [ ]:
# Test various types of queries
test_queries = [
    "What's the P/E ratio of Tesla and should I invest in it?",
    "Compare the market performance of Apple and Microsoft this month",
    "What are the top financial news headlines for NVDA?",
    "Explain the concept of dollar-cost averaging and its benefits"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"TEST QUERY {i}: {query}")
    print('='*60)
    
    try:
        response = meta_agent.invoke(query)
        print(response['output'])
    except Exception as e:
        print(f"Error: {e}")
    
    print('='*60)